# 💊 Дообучение медицинского ассистента (QLoRA)

Дообучение языковой модели для проекта **Med Assistant** — превращение
технического отчёта о препаратах в связную русскоязычную рекомендацию.

**Среда:** Google Colab, бесплатный GPU Tesla T4 (16 ГБ).

## Перед запуском
1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU**.
2. Загрузите файлы датасета `train.jsonl` и `val.jsonl`
   (из папки `finetune/dataset/` проекта) — ячейка ниже попросит их.
3. **Среда выполнения → Выполнить все**.

Обучение занимает ~20–40 минут. Готовый адаптер сохранится на ваш Google Drive.


## 1. Проверка GPU
Убедимся, что Colab выдал видеокарту. Если здесь ошибка — переключите среду на T4 GPU.

In [ ]:
!nvidia-smi
import torch
print("CUDA доступен:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ")

## 2. Установка библиотек
На Linux/T4 связка bitsandbytes+CUDA стабильна
(в отличие от Windows, где возможен краш 0xC0000005).

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate
print("Установка завершена. Если предложит перезапустить runtime — НЕ нужно, продолжайте.")

## 3. Загрузка датасета
Нажмите кнопку и выберите **оба** файла: `train.jsonl` и `val.jsonl`.

In [ ]:
from google.colab import files
import os

os.makedirs("dataset", exist_ok=True)
print("Выберите train.jsonl и val.jsonl:")
uploaded = files.upload()
for fname in uploaded:
    dst = os.path.join("dataset", fname)
    os.rename(fname, dst)
    n = sum(1 for _ in open(dst, encoding="utf-8"))
    print(f"  {fname}: {n} примеров")

## 4. Подключение Google Drive
Сюда сохранится обученный адаптер, чтобы не потерять его после закрытия сессии.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/med_assistant_lora"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Адаптер сохранится в:", SAVE_DIR)

## 5. Настройки обучения
Можно менять. По умолчанию — Llama 3 8B (на T4 16 ГБ влезает через QLoRA).
Для более быстрого обучения замените на `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
# Базовая модель (зеркало Llama без gated-доступа):
MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"
# Альтернатива (быстрее, отлично знает русский):
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 2e-4
MAX_SEQ_LEN = 1024

## 6. Загрузка и квантизация модели (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Загрузка модели (несколько минут)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("Модель загружена.")

## 7. Настройка LoRA-адаптеров
Обучаются только адаптеры (~0.5% параметров) — базовые веса заморожены.

In [ ]:
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Обучаемых параметров: {trainable:,} из {total:,} ({100*trainable/total:.2f}%)")

## 8. Подготовка датасета

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "dataset/train.jsonl",
    "validation": "dataset/val.jsonl",
})

def format_chat(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_chat)
print("Train:", len(dataset["train"]), "| Val:", len(dataset["validation"]))
print("\nПример:\n", dataset["train"][0]["text"][:400])

## 9. Обучение
По ходу будет выводиться loss. Снижение loss = модель учится.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="lora_out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
)

trainer.train()
print("Обучение завершено!")

## 10. Сохранение адаптера на Google Drive

In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Адаптер сохранён на Google Drive:", SAVE_DIR)
print("Размер:")
!du -sh "{SAVE_DIR}"

## 11. Быстрый тест дообученной модели
Сравните ответ с тем, что выдавала бы базовая модель.

In [ ]:
test_report = """Анализируемые препараты (2):

[1] Aspirin
    Действующие вещества: Acetylsalicylic acid
    Применение: профилактика инфаркта
[2] Ibuprofen
    Действующие вещества: Ibuprofen

Найденные взаимодействия (1):
  • Acetylsalicylic acid ↔ Ibuprofen
    риск или тяжесть побочных эффектов могут возрастать

Запрос пользователя: Можно ли принимать вместе?"""

SYSTEM = ("Ты — русскоязычный медицинский ассистент. На основе ТОЛЬКО предоставленных "
          "данных объясняешь простым языком назначение препарата, побочные эффекты и "
          "взаимодействия. Не выдумываешь факты. Не назначаешь дозировки. Всегда "
          "напоминаешь о необходимости консультации с врачом.")

messages = [{"role":"system","content":SYSTEM},
            {"role":"user","content":test_report}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

model.config.use_cache = True
out = model.generate(**inputs, max_new_tokens=400, temperature=0.3, top_p=0.9,
                     do_sample=True, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 12. Оценка метрик (база vs дообученная)
Считает метрики на валидации: точность фактов, доля русского, структура, дисклеймер.
Результат — таблица сравнения для презентации.

In [ ]:
import re, json

def russian_ratio(t):
    t = re.sub(r"\([^)]*\)"," ",t)
    t = re.sub(r"\b[A-Za-z][A-Za-z\-]*\s*\d+[A-Za-z0-9]*"," ",t)
    cyr=len(re.findall(r"[а-яё]",t,re.I)); lat=len(re.findall(r"[a-z]",t,re.I))
    return 1.0 if cyr+lat==0 else cyr/(cyr+lat)

def has_disclaimer(t):
    return any(k in t.lower() for k in ["врач","консульт","специалист","доктор"])

def structure_score(t):
    tl=t.lower()
    c=[any(k in tl for k in ["применя","назначен","использ","рассматрив"]),
       any(k in tl for k in ["побочн","эффект","реакци"]),
       any(k in tl for k in ["взаимодейств","сочетан","совместн","конфликт"]),
       any(k in tl for k in ["рекоменд","следует","стоит","избега","соблюда"]),
       has_disclaimer(tl)]
    return sum(c)/len(c)

def factual_accuracy(report, ans):
    conf=re.findall(r"•\s*(.+?)\s*\u2194\s*(.+)", report)
    if not conf: return None
    al=ans.lower(); hit=0
    for a,b in conf:
        ak=a.strip().lower().split()[0] if a.strip() else ""
        bk=b.strip().lower().split()[0] if b.strip() else ""
        if ak and bk and ak[:5] in al and bk[:5] in al: hit+=1
    return hit/len(conf)

def gen(m, msgs):
    p=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=tokenizer(p, return_tensors="pt").to(m.device)
    o=m.generate(**inp, max_new_tokens=400, temperature=0.3, top_p=0.9,
                 do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(o[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

N_EVAL = 40  # для скорости; можно увеличить
val = [json.loads(l) for l in open("dataset/val.jsonl", encoding="utf-8")][:N_EVAL]

def evaluate(m):
    fa,rus,st,di=[],[],[],[]
    for ex in val:
        report=ex["messages"][1]["content"]
        ans=gen(m, ex["messages"][:2])
        f=factual_accuracy(report, ans)
        if f is not None: fa.append(f)
        rus.append(russian_ratio(ans)); st.append(structure_score(ans))
        di.append(1 if has_disclaimer(ans) else 0)
    avg=lambda x: sum(x)/len(x) if x else 0
    return {"Точность фактов":avg(fa),"Доля русского":avg(rus),
            "Структура":avg(st),"Дисклеймер":avg(di)}

print(f"Оценка дообученной модели на {N_EVAL} примерах...")
m_ft = evaluate(model)
for k,v in m_ft.items():
    print(f"  {k}: {v*100:.1f}%")

### Сравнение с базовой моделью

Чтобы получить полную таблицу «база vs дообученная», в новой сессии загрузите
ту же модель **без адаптера** и прогоните `evaluate(model)` ещё раз — это и будут
числа базовой модели для сравнения.
